# Pipeline de Preparación de Corpus Bilingüe JA-ES

**Fuente:** OpenSubtitles/OPUS — `data/es-ja.tmx` (230 MB, ~1.4 M pares)

## Arquitectura del pipeline

```
 TMX ──┐
 IDS ──┴──► Etapa 1: Extracción + Filtro ──► corpus_base.jsonl
                                                     │
                       Etapa 2: Ventanas ◄───────────┘   (deque por documento)
                             │
                             ▼
                   corpus_contextual.jsonl
                             │
             Etapa 3: Validación ◄── ambos archivos
                             │
             Etapa 5: Estadísticas ──► pipeline_report.json
```

| Etapa | Descripción | Entrada | Salida |
|-------|-------------|---------|--------|
| 1 | Extracción y filtro de calidad | TMX + IDS | `corpus_base.jsonl` |
| 2 | Ventanas contextuales | `corpus_base.jsonl` | `corpus_contextual.jsonl` |
| 3 | Validación automática | ambos JSONL | errores en memoria |
| 5 | Estadísticas y reporte | ambos JSONL + stats | `pipeline_report.json` |


In [1]:
import json
import random
import re
import time
from collections import Counter, deque
from pathlib import Path
from statistics import mean
from typing import Iterator
import xml.etree.ElementTree as ET

print("✓ Librerías cargadas")


✓ Librerías cargadas


In [2]:
# ── Rutas ────────────────────────────────────────────────────────────────────
TMX_PATH         = Path("data/es-ja.tmx")
IDS_PATH         = Path("data/es-ja.txt/OpenSubtitles.es-ja.ids")
OUTPUT_DIR       = Path("data/processed")
BASE_PATH        = OUTPUT_DIR / "corpus_base.jsonl"
CONTEXTUAL_PATH  = OUTPUT_DIR / "corpus_contextual.jsonl"
PARTICLES_PATH   = OUTPUT_DIR / "corpus_filtered.jsonl"
SAMPLE_PATH      = OUTPUT_DIR / "corpus_sample.jsonl"
REPORT_PATH      = OUTPUT_DIR / "pipeline_report.json"

# ── Filtros de calidad ────────────────────────────────────────────────────────
MIN_SEGMENT_LEN = 2
MAX_SEGMENT_LEN = 1000
MAX_LEN_RATIO   = 15.0

# ── Ventana contextual ────────────────────────────────────────────────────────
WINDOW_SIZE       = 2
SAMPLES_PER_PARTICLE = 100
RANDOM_SEED          = 42
CONTEXT_SEPARATOR = " [SEP] "

# ── Partículas japonesas objetivo ─────────────────────────────────────────────
# Ordenadas de mayor a menor longitud para evitar coincidencias parciales.
# La detección ancla la partícula al fin de oración.
PARTICLES = [
    "けれども",
    "でしょうか",
    "じゃないか",
    "だろうか",
    "のかな",
    "のです",
    "んです",
    "けれど",
    "かしら",
    "でしょう",
    "だろう",
    "よね",
    "かな",
    "のだ",
    "んだ",
    "けど",
    "よ",
    "ね",
    "が",
    "ぞ",
    "ぜ",
    "さ",
    "な"
]

# ── Runtime ───────────────────────────────────────────────────────────────────
PROGRESS_EVERY = 200_000

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"TMX  : {TMX_PATH}  ({TMX_PATH.stat().st_size / 1e6:.0f} MB)")
print(f"Base : {BASE_PATH}")
print(f"Ctx  : {CONTEXTUAL_PATH}")
print(f"Part : {PARTICLES_PATH}")
print(f"\nFiltros  min={MIN_SEGMENT_LEN}  max={MAX_SEGMENT_LEN}  ratio={MAX_LEN_RATIO}")
print(f"Ventana  size={WINDOW_SIZE}  sep={CONTEXT_SEPARATOR!r}")
print(f"Partículas objetivo: {len(PARTICLES)}")


TMX  : data/es-ja.tmx  (241 MB)
Base : data/processed/corpus_base.jsonl
Ctx  : data/processed/corpus_contextual.jsonl
Part : data/processed/corpus_filtered.jsonl

Filtros  min=2  max=1000  ratio=15.0
Ventana  size=2  sep=' [SEP] '
Partículas objetivo: 23


## Funciones auxiliares

Parsers de entrada, limpieza de texto y filtros de calidad.
Estas funciones son compartidas por todas las etapas del pipeline.


In [3]:
# ── Expresiones regulares ─────────────────────────────────────────────────────
_HTML_TAG  = re.compile(r"<[^>]+>")
_TIMECODE  = re.compile(r"\d{1,2}:\d{2}:\d{2}")
_MULTI_SPC = re.compile(r"[ \t]+")
_MULTI_NL  = re.compile(r"\n{2,}")

# Patrón de detección de partículas: ancla en fin de oración.
# Permite puntuación final opcional (。！？…」) después de la partícula.
_SENT_PUNCT = r"[。！？…」\s]*"


def _build_particle_patterns(particles: list[str]) -> dict:
    return {
        p: re.compile(re.escape(p) + _SENT_PUNCT + r"$")
        for p in particles
    }


# Se construye una vez al cargar la celda; se reconstruye si PARTICLES cambia.
_PARTICLE_RE: dict = _build_particle_patterns(PARTICLES)


def detect_particles(text: str) -> tuple[Optional[str], list[str]]:
    """
    Detecta partículas finales en text.

    Estrategia: verificar si la partícula aparece justo antes del fin de la
    cadena (opcionalmente precedida por puntuación final).  Las partículas se
    evalúan en orden de mayor a menor longitud para evitar que 'けれど' oculte
    a 'けれども'.

    Retorna (primera_detectada | None, [todas_detectadas]).
    """
    detected = [p for p in PARTICLES if _PARTICLE_RE[p].search(text)]
    return (detected[0] if detected else None), detected


def iter_tmx(path: Path) -> Iterator[tuple[str, str]]:
    """Parsea el TMX por streaming con iterparse; yields (es_text, ja_text)."""
    for _, elem in ET.iterparse(str(path), events=["end"]):
        if elem.tag != "tu":
            continue
        es = ja = ""
        for tuv in elem.findall("tuv"):
            lang = (tuv.get("{http://www.w3.org/XML/1998/namespace}lang")
                    or tuv.get("xml:lang", ""))
            seg  = tuv.find("seg")
            text = (seg.text or "") if seg is not None else ""
            if lang == "es":
                es = text
            elif lang == "ja":
                ja = text
        elem.clear()
        yield es, ja


def iter_ids(path: Path) -> Iterator[str]:
    """Yields document_id alineado posicionalmente con iter_tmx."""
    with open(path, encoding="utf-8") as f:
        for line in f:
            es_path = line.split("\t", 1)[0].strip()
            yield es_path.removeprefix("es/").removesuffix(".xml.gz")


def clean_segment(text: str) -> str:
    """Normaliza espacios; preserva todos los caracteres y puntuación originales."""
    text = _MULTI_SPC.sub(" ", text)
    text = _MULTI_NL.sub("\n", text)
    return text.strip()


def passes_quality_filter(es: str, ja: str) -> bool:
    """True si el par cumple todos los umbrales de calidad configurados."""
    if not es or not ja:
        return False
    le, lj = len(es), len(ja)
    if le < MIN_SEGMENT_LEN or lj < MIN_SEGMENT_LEN:
        return False
    if le > MAX_SEGMENT_LEN or lj > MAX_SEGMENT_LEN:
        return False
    if max(le, lj) / min(le, lj) > MAX_LEN_RATIO:
        return False
    if _HTML_TAG.search(es) or _HTML_TAG.search(ja):
        return False
    if _TIMECODE.search(es) or _TIMECODE.search(ja):
        return False
    return True


# Verificación rápida de detect_particles
_test_cases = [
    ("今から話してあげようね。",    "ね"),
    ("本当にそうですよ",            "よ"),
    ("行くかな",                     "かな"),
    ("けれどもそれは違うよ",         "よ"),
    ("わからないけれども",           "けれども"),
    ("これは難しいんです",           "んです"),
    ("どうしようかな…",             "かな"),
    ("これは普通の文章です。",       None),
]
ok = all(detect_particles(t)[0] == expected for t, expected in _test_cases)
print(f"✓ detect_particles() lista  (test {'OK' if ok else 'FAIL'})")
if not ok:
    for t, expected in _test_cases:
        got, _ = detect_particles(t)
        if got != expected:
            print(f"  FAIL: {t!r}  expected={expected!r}  got={got!r}")
print("✓ Helpers listos")


✓ detect_particles() lista  (test OK)
✓ Helpers listos


## Etapa 1 — corpus Base

Extrae pares ES-JA del TMX, aplica filtros de calidad y escribe `corpus_base.jsonl`.

### Esquema de salida

```json
{
  "document_id":     "1920/10323/5829947",
  "sequence_id":     0,
  "doc_sequence_id": 0,
  "japanese_text":   "カリガリ博士の小屋",
  "spanish_text":    "EL GABINETE DEL DR. CALIGARI",
  "sentiment":       null,
  "intent":          null
}
```

- `sequence_id` — contador global de registros guardados (no se reinicia)
- `doc_sequence_id` — posición dentro del documento (se reinicia a 0 en cada nuevo `document_id`)
- `sentiment` / `intent` — placeholders para fases posteriores


In [4]:
def run_stage1(tmx_path: Path, ids_path: Path, output_path: Path) -> dict:
    """
    Etapa 1: genera corpus_base.jsonl por streaming.

    Lee TMX e IDS en paralelo (mismo número de registros, alineados por posición).
    Aplica filtros de calidad y asigna sequence_id global y doc_sequence_id por documento.
    """
    stats       = {"total_read": 0, "total_kept": 0, "total_dropped": 0}
    current_doc = None
    doc_seq     = -1
    global_seq  = 0
    start       = time.perf_counter()

    with open(output_path, "w", encoding="utf-8") as out:
        for (es_raw, ja_raw), doc_id in zip(iter_tmx(tmx_path), iter_ids(ids_path)):
            stats["total_read"] += 1

            es = clean_segment(es_raw)
            ja = clean_segment(ja_raw)

            if not passes_quality_filter(es, ja):
                stats["total_dropped"] += 1
                continue

            if doc_id != current_doc:
                current_doc = doc_id
                doc_seq = 0
            else:
                doc_seq += 1

            record = {
                "document_id":     doc_id,
                "sequence_id":     global_seq,
                "doc_sequence_id": doc_seq,
                "japanese_text":   ja,
                "spanish_text":    es,
                "sentiment":       None,
                "intent":          None,
            }
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            global_seq += 1
            stats["total_kept"] += 1

            if stats["total_read"] % PROGRESS_EVERY == 0:
                elapsed = time.perf_counter() - start
                print(f"  [{stats['total_read']:>9,}]  "
                      f"kept={stats['total_kept']:,}  "
                      f"dropped={stats['total_dropped']:,}  "
                      f"rate={stats['total_read']/elapsed:,.0f} rec/s")

    elapsed = time.perf_counter() - start
    stats["elapsed_s"]     = round(elapsed, 1)
    stats["drop_rate_pct"] = round(100 * stats["total_dropped"] / stats["total_read"], 2)
    return stats


print("✓ run_stage1() lista")


✓ run_stage1() lista


In [5]:
print(f"Etapa 1: generando {BASE_PATH} ...\n")
stage1_stats = run_stage1(TMX_PATH, IDS_PATH, BASE_PATH)

print(f"\n{'─'*52}")
print(f"  Pares leídos   : {stage1_stats['total_read']:>10,}")
print(f"  Guardados      : {stage1_stats['total_kept']:>10,}")
print(f"  Descartados    : {stage1_stats['total_dropped']:>10,}  ({stage1_stats['drop_rate_pct']}%)")
print(f"  Tiempo         : {stage1_stats['elapsed_s']}s")
print(f"  Tamaño archivo : {BASE_PATH.stat().st_size / 1e6:.1f} MB")

print("\nMuestra — primeros 3 registros:")
with open(BASE_PATH, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        r = json.loads(line)
        print(f"\n  seq={r['sequence_id']}  doc_seq={r['doc_sequence_id']}"
              f"  doc={r['document_id']}")
        print(f"  ES: {r['spanish_text']}")
        print(f"  JA: {r['japanese_text']}")


Etapa 1: generando data/processed/corpus_base.jsonl ...

  [  200,000]  kept=199,460  dropped=540  rate=58,967 rec/s
  [  400,000]  kept=398,940  dropped=1,060  rate=58,431 rec/s
  [  600,000]  kept=598,579  dropped=1,421  rate=57,945 rec/s
  [  800,000]  kept=798,187  dropped=1,813  rate=58,051 rec/s
  [1,000,000]  kept=997,914  dropped=2,086  rate=55,466 rec/s
  [1,200,000]  kept=1,197,652  dropped=2,348  rate=55,615 rec/s

────────────────────────────────────────────────────
  Pares leídos   :  1,314,860
  Guardados      :  1,312,338
  Descartados    :      2,522  (0.19%)
  Tiempo         : 23.8s
  Tamaño archivo : 316.8 MB

Muestra — primeros 3 registros:

  seq=0  doc_seq=0  doc=1920/10323/5829947
  ES: EL GABINETE DEL DR. CALIGARI
  JA: カリガリ博士の小屋

  seq=1  doc_seq=1  doc=1920/10323/5829947
  ES: ACTO I
  JA: 幕１

  seq=2  doc_seq=2  doc=1920/10323/5829947
  ES: Hay fantasmas... están por todas partes a nuestro alrededor. Me han expulsado de casa y del hogar, lejos de mi mujer e hi

## Etapa 2 — Ventanas Contextuales con Trazabilidad de Partículas

Lee `corpus_base.jsonl` y genera dos archivos en un único paso:

| Archivo | Contenido |
|---------|-----------|
| `corpus_contextual.jsonl` | Todos los registros |
| `corpus_filtered.jsonl`  | Solo registros donde `particle != null` |

La ventana contextual no se modifica. La detección de partículas opera
únicamente sobre `current_ja` como metadato de trazabilidad.

### Esquema de salida

```json
{
  "document_id":        "1920/10323/5829947",
  "target_sequence":    25,
  "context_ja":         "カリガリ博士の小屋 [SEP] 幕１",
  "current_ja":         "僕の婚約者だね",
  "context_es":         "EL GABINETE DEL DR. CALIGARI [SEP] ACTO I",
  "current_es":         "Es mi prometida...",
  "particle":           "ね",
  "particles_detected": ["ね"],
  "sentiment":          null,
  "intent":             null
}
```


In [6]:
def run_stage2(
    base_path: Path,
    output_path: Path,
    particles_path: Path,
) -> dict:
    """
    Etapa 2: genera corpus_contextual.jsonl y corpus_filtered.jsonl
    en un único paso de lectura sobre corpus_base.jsonl.

    corpus_contextual.jsonl  — todos los registros contextuales
    corpus_filtered.jsonl   — subconjunto donde particle != null

    La lógica de ventanización es idéntica a la versión original.
    La detección de partículas opera únicamente sobre current_ja
    como metadato de trazabilidad; no altera el contenido contextual.
    """
    stats = {
        "total_contextual": 0,
        "with_particle":    0,
        "without_particle": 0,
        "particle_freq":    Counter(),
    }
    buffer: deque = deque(maxlen=WINDOW_SIZE)
    current_doc   = None
    start         = time.perf_counter()

    with open(base_path, encoding="utf-8") as in_f, \
         open(output_path,    "w", encoding="utf-8") as out_all, \
         open(particles_path, "w", encoding="utf-8") as out_part:

        for line in in_f:
            rec    = json.loads(line)
            doc_id = rec["document_id"]

            # ── Límite documental ─────────────────────────────────────────────
            if doc_id != current_doc:
                buffer.clear()
                current_doc = doc_id

            # ── Ventana contextual (lógica original intacta) ──────────────────
            ctx_list = list(buffer)

            # ── Detección de partículas (solo trazabilidad) ───────────────────
            particle, particles_detected = detect_particles(rec["japanese_text"])

            record = {
                "document_id":        doc_id,
                "target_sequence":    rec["sequence_id"],
                "context_ja":         CONTEXT_SEPARATOR.join(
                                          r["japanese_text"] for r in ctx_list),
                "current_ja":         rec["japanese_text"],
                "context_es":         CONTEXT_SEPARATOR.join(
                                          r["spanish_text"] for r in ctx_list),
                "current_es":         rec["spanish_text"],
                "particle":           particle,
                "particles_detected": particles_detected,
                "sentiment":          None,
                "intent":             None,
            }

            line_out = json.dumps(record, ensure_ascii=False) + "\n"
            out_all.write(line_out)
            if particle is not None:
                out_part.write(line_out)

            # ── Buffer y estadísticas ─────────────────────────────────────────
            buffer.append(rec)
            stats["total_contextual"] += 1
            if particle is not None:
                stats["with_particle"] += 1
                for p in particles_detected:
                    stats["particle_freq"][p] += 1
            else:
                stats["without_particle"] += 1

            if stats["total_contextual"] % PROGRESS_EVERY == 0:
                elapsed = time.perf_counter() - start
                print(f"  [{stats['total_contextual']:>9,}]  "
                      f"con_partícula={stats['with_particle']:,}  "
                      f"rate={stats['total_contextual']/elapsed:,.0f} rec/s")

    elapsed = time.perf_counter() - start
    stats["elapsed_s"]    = round(elapsed, 1)
    stats["particle_pct"] = round(
        100 * stats["with_particle"] / max(stats["total_contextual"], 1), 2
    )
    stats["particle_freq"] = dict(
        sorted(stats["particle_freq"].items(), key=lambda x: -x[1])
    )
    return stats


print("✓ run_stage2() lista")


✓ run_stage2() lista


In [7]:
print(f"Etapa 2: generando archivos contextuales ...\n")
stage2_stats = run_stage2(BASE_PATH, CONTEXTUAL_PATH, PARTICLES_PATH)

print(f"\n{'─'*52}")
print(f"  Total contextuales    : {stage2_stats['total_contextual']:>10,}")
print(f"  Con partícula         : {stage2_stats['with_particle']:>10,}"
      f"  ({stage2_stats['particle_pct']}%)")
print(f"  Sin partícula         : {stage2_stats['without_particle']:>10,}")
print(f"  Tiempo                : {stage2_stats['elapsed_s']}s")
print(f"  corpus_contextual    : {CONTEXTUAL_PATH.stat().st_size / 1e6:.1f} MB")
print(f"  corpus_filtered     : {PARTICLES_PATH.stat().st_size / 1e6:.1f} MB")

print("\nFrecuencia de partículas detectadas:")
total_ctx = stage2_stats["total_contextual"]
for p, cnt in stage2_stats["particle_freq"].items():
    pct = 100 * cnt / total_ctx
    bar = "█" * max(1, int(pct * 2))
    print(f"  {p:<10} {cnt:>8,}  {pct:5.1f}%  {bar}")

# Muestra con partícula
print("\nMuestra — corpus_filtered.jsonl (primer registro):")
with open(PARTICLES_PATH, encoding="utf-8") as f:
    print(json.dumps(json.loads(f.readline()), ensure_ascii=False, indent=2))

# Muestra sin partícula
print("\nMuestra — corpus_contextual.jsonl sin partícula:")
with open(CONTEXTUAL_PATH, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r["particle"] is None:
            print(json.dumps(r, ensure_ascii=False, indent=2))
            break


Etapa 2: generando archivos contextuales ...

  [  200,000]  con_partícula=40,716  rate=56,071 rec/s
  [  400,000]  con_partícula=79,403  rate=58,611 rec/s
  [  600,000]  con_partícula=117,468  rate=59,645 rec/s
  [  800,000]  con_partícula=154,299  rate=60,462 rec/s
  [1,000,000]  con_partícula=188,873  rate=60,791 rec/s
  [1,200,000]  con_partícula=223,481  rate=60,966 rec/s

────────────────────────────────────────────────────
  Total contextuales    :  1,312,338
  Con partícula         :    241,780  (18.42%)
  Sin partícula         :  1,070,558
  Tiempo                : 21.5s
  corpus_contextual    : 615.4 MB
  corpus_filtered     : 115.0 MB

Frecuencia de partículas detectadas:
  よ            70,605    5.4%  ██████████
  んだ           35,639    2.7%  █████
  ね            34,707    2.6%  █████
  な            34,439    2.6%  █████
  ぞ            15,454    1.2%  ██
  が            14,544    1.1%  ██
  さ             7,730    0.6%  █
  だろう           6,141    0.5%  █
  けど            4,583

## Etapa 4 — Muestreo Estratificado por Partícula

Lee `corpus_filtered.jsonl` y extrae hasta `SAMPLES_PER_PARTICLE` registros por partícula usando **reservoir sampling** (un solo paso, sin sesgo de posición). El resultado se guarda en `corpus_sample.jsonl`.

In [8]:
def run_stage_sample(
    particles_path: Path,
    sample_path: Path,
    samples_per_particle: int = SAMPLES_PER_PARTICLE,
) -> dict:
    """
    Etapa 4: muestrea corpus_filtered.jsonl con reservoir sampling.

    Por cada partícula se mantiene un reservorio de tamaño samples_per_particle.
    Al procesar el registro k-ésimo de una partícula (k > samples_per_particle),
    se reemplaza un elemento aleatorio del reservorio con probabilidad
    samples_per_particle/k, garantizando muestreo uniforme sin cargar todo el
    archivo en memoria.
    """
    random.seed(RANDOM_SEED)
    reservoirs: dict[str, list] = {p: [] for p in PARTICLES}
    counts: Counter = Counter()

    with open(particles_path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            p   = rec["particle"]
            if p not in reservoirs:
                continue
            if not rec.get("context_ja"):
                continue
            counts[p] += 1
            n   = counts[p]
            res = reservoirs[p]
            if len(res) < samples_per_particle:
                res.append(rec)
            else:
                j = random.randint(0, n - 1)
                if j < samples_per_particle:
                    res[j] = rec

    # Escribir: agrupar por partícula, ordenar por target_sequence dentro de cada grupo
    total = 0
    with open(sample_path, "w", encoding="utf-8") as out:
        for p in PARTICLES:
            for rec in sorted(reservoirs[p], key=lambda r: r["target_sequence"]):
                out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                total += 1

    return {
        "total_in_particles_file": sum(counts.values()),
        "total_sampled":           total,
        "per_particle_sampled":    {p: len(reservoirs[p]) for p in PARTICLES},
        "per_particle_available":  dict(counts),
    }


print("\u2713 run_stage_sample() lista")


✓ run_stage_sample() lista


In [9]:
print(f"Etapa 4: muestreando {PARTICLES_PATH} -> {SAMPLE_PATH} ...\n")
sample_stats = run_stage_sample(PARTICLES_PATH, SAMPLE_PATH)

print(f"  Total en corpus_filtered : {sample_stats['total_in_particles_file']:>8,}")
print(f"  Total en muestra           : {sample_stats['total_sampled']:>8,}")
print(f"  Tama\u00f1o archivo               : {SAMPLE_PATH.stat().st_size / 1e6:.2f} MB")
print()
print(f"  {'Part\u00edcula':<12} {'Disponibles':>12} {'Muestreados':>12}")
print(f"  {'--------':<12} {'----------':>12} {'----------':>12}")
for p in PARTICLES:
    avail   = sample_stats['per_particle_available'].get(p, 0)
    sampled = sample_stats['per_particle_sampled'].get(p, 0)
    flag    = " \u26a0 insuficiente" if avail < SAMPLES_PER_PARTICLE else ""
    print(f"  {p:<12} {avail:>12,} {sampled:>12,}{flag}")


Etapa 4: muestreando data/processed/corpus_filtered.jsonl -> data/processed/corpus_sample.jsonl ...

  Total en corpus_filtered :  241,468
  Total en muestra           :    2,169
  Tamaño archivo               : 1.10 MB

  Partícula     Disponibles  Muestreados
  --------       ----------   ----------
  けれども                    8            8 ⚠ insuficiente
  でしょうか                 434          100
  じゃないか               1,359          100
  だろうか                  272          100
  のかな                   553          100
  のです                 1,510          100
  んです                 3,280          100
  けれど                    61           61 ⚠ insuficiente
  かしら                 1,833          100
  でしょう                3,035          100
  だろう                 6,138          100
  よね                  2,488          100
  かな                  3,768          100
  のだ                  2,941          100
  んだ                 35,599          100
  けど                  4,578          100
  よ        

## Etapa 3 — Validación Automática

Verifica la integridad de ambos corpuss sin cargarlos completamente en memoria.

### Checks sobre `corpus_base.jsonl`

| Check | Qué detecta |
|-------|-------------|
| `empty_japanese` | Segmentos JA vacíos tras el filtro |
| `empty_spanish` | Segmentos ES vacíos tras el filtro |
| `sequence_gap` | Saltos en `sequence_id` (indica pérdida de registros) |
| `doc_sequence_wrong` | `doc_sequence_id` no es contiguo dentro del documento |

### Checks sobre `corpus_contextual.jsonl`

| Check | Qué detecta |
|-------|-------------|
| `empty_current_ja` | Texto JA actual vacío |
| `empty_current_es` | Texto ES actual vacío |
| `sep_count_mismatch` | Ventana completa pero número de partes ≠ `WINDOW_SIZE` |
| `complete_flag_inconsistent` | `context_complete=True` con contexto vacío |


In [10]:
def validate_base(path: Path) -> dict:
    """Valida corpus_base.jsonl de forma streaming."""
    errors = {
        "empty_japanese":     [],
        "empty_spanish":      [],
        "sequence_gap":       [],
        "doc_sequence_wrong": [],
    }
    prev_seq     = -1
    prev_doc     = None
    prev_doc_seq = -1

    with open(path, encoding="utf-8") as f:
        for line in f:
            r       = json.loads(line)
            seq     = r["sequence_id"]
            doc_id  = r["document_id"]
            doc_seq = r["doc_sequence_id"]

            if not r.get("japanese_text", "").strip():
                errors["empty_japanese"].append(seq)
            if not r.get("spanish_text", "").strip():
                errors["empty_spanish"].append(seq)

            if seq != prev_seq + 1:
                errors["sequence_gap"].append({"expected": prev_seq + 1, "got": seq})
            prev_seq = seq

            if doc_id == prev_doc:
                if doc_seq != prev_doc_seq + 1:
                    errors["doc_sequence_wrong"].append(
                        {"doc": doc_id, "expected": prev_doc_seq + 1, "got": doc_seq}
                    )
            else:
                if doc_seq != 0:
                    errors["doc_sequence_wrong"].append(
                        {"doc": doc_id, "expected": 0, "got": doc_seq}
                    )
            prev_doc     = doc_id
            prev_doc_seq = doc_seq

    return {
        "errors":          errors,
        "total_errors":    sum(len(v) for v in errors.values()),
        "checked_records": prev_seq + 1,
    }


def validate_contextual(path: Path) -> dict:
    """
    Valida corpus_contextual.jsonl de forma streaming.

    Checks:
      - Textos actuales no vacíos.
      - Formato de partículas consistente:
          * particle None  ↔  particles_detected == []
          * particle str   ↔  particle in particles_detected
      - Separadores en context_ja coherentes con el contenido.
    """
    errors = {
        "empty_current_ja":          [],
        "empty_current_es":          [],
        "particle_field_mismatch":   [],   # particle vs particles_detected inconsistentes
        "context_sep_unexpected":    [],   # SEP en contexto vacío
    }
    sep     = CONTEXT_SEPARATOR.strip()
    checked = 0

    with open(path, encoding="utf-8") as f:
        for line in f:
            r        = json.loads(line)
            seq      = r["target_sequence"]
            ctx_ja   = r["context_ja"]
            particle = r["particle"]
            detected = r["particles_detected"]

            if not r.get("current_ja", "").strip():
                errors["empty_current_ja"].append(seq)
            if not r.get("current_es", "").strip():
                errors["empty_current_es"].append(seq)

            # Coherencia entre particle y particles_detected
            if particle is None and detected:
                errors["particle_field_mismatch"].append(
                    {"seq": seq, "issue": "particle=null but particles_detected non-empty"}
                )
            if particle is not None and particle not in detected:
                errors["particle_field_mismatch"].append(
                    {"seq": seq, "issue": f"particle={particle!r} not in particles_detected={detected}"}
                )

            # SEP en contexto vacío indica error lógico
            if not ctx_ja.strip() and sep in ctx_ja:
                errors["context_sep_unexpected"].append(seq)

            checked += 1

    return {
        "errors":          errors,
        "total_errors":    sum(len(v) for v in errors.values()),
        "checked_records": checked,
    }


print("✓ validate_base() y validate_contextual() listas")


✓ validate_base() y validate_contextual() listas


In [11]:
print("Etapa 3: validando corpus_base.jsonl ...")
val_base = validate_base(BASE_PATH)
print(f"  Registros verificados : {val_base['checked_records']:,}")
print(f"  Errores totales       : {val_base['total_errors']}")
for name, lst in val_base["errors"].items():
    icon = "✓" if not lst else "⚠"
    note = f"{len(lst)} ocurrencias   muestra={lst[:2]}" if lst else "OK"
    print(f"    {icon} {name:<28} {note}")

print()
print("Etapa 3: validando corpus_contextual.jsonl ...")
val_ctx = validate_contextual(CONTEXTUAL_PATH)
print(f"  Registros verificados : {val_ctx['checked_records']:,}")
print(f"  Errores totales       : {val_ctx['total_errors']}")
for name, lst in val_ctx["errors"].items():
    icon = "✓" if not lst else "⚠"
    note = f"{len(lst)} ocurrencias   muestra={lst[:2]}" if lst else "OK"
    print(f"    {icon} {name:<32} {note}")

validation_results = {"base": val_base, "contextual": val_ctx}


Etapa 3: validando corpus_base.jsonl ...
  Registros verificados : 1,312,338
  Errores totales       : 0
    ✓ empty_japanese               OK
    ✓ empty_spanish                OK
    ✓ sequence_gap                 OK
    ✓ doc_sequence_wrong           OK

Etapa 3: validando corpus_contextual.jsonl ...
  Registros verificados : 1,312,338
  Errores totales       : 0
    ✓ empty_current_ja                 OK
    ✓ empty_current_es                 OK
    ✓ particle_field_mismatch          OK
    ✓ context_sep_unexpected           OK


## Etapa 5 — Estadísticas y Reporte

Calcula métricas completas del corpus y guarda `pipeline_report.json`.
Lee `corpus_base.jsonl` una única vez para distribuciones de longitud y tamaño de documentos.


In [12]:
def percentile_summary(data: list) -> dict:
    """Resumen estadístico de una lista de números (sin dependencias externas)."""
    if not data:
        return {}
    s = sorted(data)
    n = len(s)
    return {
        "min":  s[0],
        "p25":  s[n // 4],
        "p50":  s[n // 2],
        "p75":  s[3 * n // 4],
        "p95":  s[int(0.95 * n)],
        "max":  s[-1],
        "mean": round(mean(s), 1),
    }


def compute_statistics(
    base_path: Path,
    contextual_path: Path,
    s1: dict,
    s2: dict,
    val: dict,
) -> dict:
    """
    Etapa 5: calcula distribuciones de longitud, tamaño de documentos
    y métricas de detección de partículas.
    """
    doc_counts: Counter = Counter()
    len_es:     list    = []
    len_ja:     list    = []

    print("  Leyendo corpus_base para distribuciones ...")
    with open(base_path, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            doc_counts[r["document_id"]] += 1
            len_es.append(len(r["spanish_text"]))
            len_ja.append(len(r["japanese_text"]))

    size_buckets = {"1-10": 0, "11-50": 0, "51-200": 0, "201-500": 0, "500+": 0}
    for size in doc_counts.values():
        if   size <=  10: size_buckets["1-10"]    += 1
        elif size <=  50: size_buckets["11-50"]   += 1
        elif size <= 200: size_buckets["51-200"]  += 1
        elif size <= 500: size_buckets["201-500"] += 1
        else:             size_buckets["500+"]    += 1

    report = {
        "corpus": {
            "total_documents":     len(doc_counts),
            "total_pairs_read":    s1["total_read"],
            "total_pairs_kept":    s1["total_kept"],
            "total_pairs_dropped": s1["total_dropped"],
            "drop_rate_pct":       s1["drop_rate_pct"],
        },
        "contextual": {
            "total_examples":   s2["total_contextual"],
            "with_particle":    s2["with_particle"],
            "without_particle": s2["without_particle"],
            "particle_pct":     s2["particle_pct"],
            "window_size":      WINDOW_SIZE,
            "separator":        CONTEXT_SEPARATOR.strip(),
        },
        "particle_frequency":             s2["particle_freq"],
        "document_size_distribution":     size_buckets,
        "length_distribution_es":         percentile_summary(len_es),
        "length_distribution_ja":         percentile_summary(len_ja),
        "validation": {
            "base_total_errors":       val["base"]["total_errors"],
            "contextual_total_errors": val["contextual"]["total_errors"],
        },
        "output_files": {
            "corpus_base_mb":       round(base_path.stat().st_size / 1e6, 1),
            "corpus_contextual_mb":  round(contextual_path.stat().st_size / 1e6, 1),
            "corpus_filtered_mb":   round(PARTICLES_PATH.stat().st_size / 1e6, 1),
        },
        "timing": {
            "stage1_elapsed_s": s1["elapsed_s"],
            "stage2_elapsed_s": s2["elapsed_s"],
        },
    }
    return report


print("✓ compute_statistics() lista")


✓ compute_statistics() lista


In [13]:
print("Etapa 5: calculando estadísticas ...\n")
report = compute_statistics(
    BASE_PATH, CONTEXTUAL_PATH,
    stage1_stats, stage2_stats, validation_results
)

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(f"Reporte guardado: {REPORT_PATH}\n")

print(json.dumps(report, ensure_ascii=False, indent=2))


Etapa 5: calculando estadísticas ...

  Leyendo corpus_base para distribuciones ...
Reporte guardado: data/processed/pipeline_report.json

{
  "corpus": {
    "total_documents": 1832,
    "total_pairs_read": 1314860,
    "total_pairs_kept": 1312338,
    "total_pairs_dropped": 2522,
    "drop_rate_pct": 0.19
  },
  "contextual": {
    "total_examples": 1312338,
    "with_particle": 241780,
    "without_particle": 1070558,
    "particle_pct": 18.42,
    "window_size": 2,
    "separator": "[SEP]"
  },
  "particle_frequency": {
    "よ": 70605,
    "んだ": 35639,
    "ね": 34707,
    "な": 34439,
    "ぞ": 15454,
    "が": 14544,
    "さ": 7730,
    "だろう": 6141,
    "けど": 4583,
    "かな": 4331,
    "んです": 3281,
    "ぜ": 3192,
    "でしょう": 3037,
    "のだ": 2941,
    "よね": 2490,
    "かしら": 1835,
    "のです": 1512,
    "じゃないか": 1363,
    "のかな": 553,
    "でしょうか": 435,
    "だろうか": 273,
    "けれど": 61,
    "けれども": 8
  },
  "document_size_distribution": {
    "1-10": 0,
    "11-50": 9,
    "51-200": 56,
    "2

## Resumen final del pipeline

In [14]:
c  = report["corpus"]
x  = report["contextual"]
ve = report["validation"]

status_b = "✓ OK" if ve["base_total_errors"] == 0 else f"⚠  {ve['base_total_errors']} errores"
status_c = "✓ OK" if ve["contextual_total_errors"] == 0 else f"⚠  {ve['contextual_total_errors']} errores"

print("=" * 60)
print("  PIPELINE COMPLETO")
print("=" * 60)
print(f"\n  Documentos únicos     : {c['total_documents']:>10,}")
print(f"  Pares base (guardados): {c['total_pairs_kept']:>10,}")
print(f"  Pares descartados     : {c['total_pairs_dropped']:>10,}  ({c['drop_rate_pct']}%)")
print(f"  Ejemplos contextuales : {x['total_examples']:>10,}")
print()
print(f"  Partículas detectadas : {x['with_particle']:>10,}  ({x['particle_pct']}%)")
print(f"  Sin partícula         : {x['without_particle']:>10,}")
print()
print("  Top partículas:")
for p, cnt in list(report["particle_frequency"].items())[:8]:
    pct = 100 * cnt / max(x["total_examples"], 1)
    print(f"    {p:<10} {cnt:>10,}  ({pct:.1f}%)")
print()
print("  Distribución de longitud (ES):")
d = report["length_distribution_es"]
print(f"    p25={d['p25']}  p50={d['p50']}  p75={d['p75']}  p95={d['p95']}  media={d['mean']}")
print("  Distribución de longitud (JA):")
d = report["length_distribution_ja"]
print(f"    p25={d['p25']}  p50={d['p50']}  p75={d['p75']}  p95={d['p95']}  media={d['mean']}")
print()
print("  Archivos generados:")
print(f"    {BASE_PATH}")
print(f"      → {report['output_files']['corpus_base_mb']} MB")
print(f"    {CONTEXTUAL_PATH}")
print(f"      → {report['output_files']['corpus_contextual_mb']} MB")
print(f"    {REPORT_PATH}")
print()
print("  Validación:")
print(f"    corpus_base        : {status_b}")
print(f"    corpus_contextual  : {status_c}")
print()
print("  Tiempo total:")
print(f"    Etapa 1 (TMX → base)      : {report['timing']['stage1_elapsed_s']}s")
print(f"    Etapa 2 (base → ctx)      : {report['timing']['stage2_elapsed_s']}s")


  PIPELINE COMPLETO

  Documentos únicos     :      1,832
  Pares base (guardados):  1,312,338
  Pares descartados     :      2,522  (0.19%)
  Ejemplos contextuales :  1,312,338

  Partículas detectadas :    241,780  (18.42%)
  Sin partícula         :  1,070,558

  Top partículas:
    よ              70,605  (5.4%)
    んだ             35,639  (2.7%)
    ね              34,707  (2.6%)
    な              34,439  (2.6%)
    ぞ              15,454  (1.2%)
    が              14,544  (1.1%)
    さ               7,730  (0.6%)
    だろう             6,141  (0.5%)

  Distribución de longitud (ES):
    p25=20  p50=31  p75=48  p95=87  media=37.4
  Distribución de longitud (JA):
    p25=8  p50=12  p75=18  p95=32  media=14.1

  Archivos generados:
    data/processed/corpus_base.jsonl
      → 316.8 MB
    data/processed/corpus_contextual.jsonl
      → 615.4 MB
    data/processed/pipeline_report.json

  Validación:
    corpus_base        : ✓ OK
    corpus_contextual  : ✓ OK

  Tiempo total:
    Etapa 1 (TMX 